# dko-3dgs — pipeline local

Corre el pipeline completo en esta máquina (`~/dko-3dgs`), reutilizando los scripts del repo:

```
video local → candidates/ (ffmpeg) → select_sharp.py → data/input/
            → run_colmap.sh (CPU) o run_hloc.py (GPU) → data/images + data/sparse/0
            → gaussian-splatting/train.py → output/dko3d/
```

**Requisitos ya instalados en esta máquina:** COLMAP (build CPU), hloc + pycolmap, ffmpeg, y el clon de `gaussian-splatting/` con sus submódulos CUDA.

⚠️ Los directorios `candidates/`, `data/`, `hloc_out/` y `output/` ya tienen los resultados de la corrida anterior. Las celdas marcadas con ⚠️ los sobrescriben — sáltalas si solo quieres re-entrenar.

## 0. Configuración

In [ ]:
from pathlib import Path

ROOT = Path.home() / "dko-3dgs"

# Ruta al video de entrada (ajústala). Solo se usa si vas a re-extraer frames.
VIDEO = ""  # p. ej. "/home/leo/videos/dko.mp4"

TARGET_IMAGES = 240   # cuántas imágenes nítidas conservar aprox.
ITERATIONS = 7000     # iteraciones de entrenamiento 3DGS

print("ROOT:", ROOT)
print("candidates:", len(list((ROOT / 'candidates').glob('*.jpg'))), "frames")
print("data/input:", len(list((ROOT / 'data' / 'input').glob('*.jpg'))), "imágenes")

## 1. ⚠️ Extraer frames del video

Solo si partes de un video nuevo. Sobrescribe `candidates/`.

In [ ]:
assert VIDEO, "Define VIDEO en la celda de configuración antes de correr esta celda"

import shutil
cand = ROOT / "candidates"
if cand.exists():
    shutil.rmtree(cand)
cand.mkdir()

!ffmpeg -hide_banner -loglevel error -i "{VIDEO}" -qscale:v 2 "{cand}/%05d.jpg"
print(len(list(cand.glob('*.jpg'))), "frames extraídos")

## 2. ⚠️ Seleccionar frames nítidos

Usa `select_sharp.py` del repo (varianza del Laplaciano por ventana). Sobrescribe `data/input/`.

In [ ]:
import shutil

n_frames = len(list((ROOT / 'candidates').glob('*.jpg')))
WIN = max(1, n_frames // TARGET_IMAGES)
print(f"{n_frames} frames, ventana = {WIN}")

inp = ROOT / "data" / "input"
if inp.exists():
    shutil.rmtree(inp)

!python "{ROOT}/select_sharp.py" "{ROOT}/candidates" "{inp}" {WIN}

## 3. Structure from Motion — elige UNA de las dos opciones

### Opción A: hloc (GPU) — recomendada, minutos

ALIKED + LightGlue + mapper de pycolmap (`run_hloc.py`). Luego se undistorsiona al layout `data/images` + `data/sparse/0` que espera 3DGS.

In [ ]:
import shutil

hloc_out = ROOT / "hloc_out"
if hloc_out.exists():
    shutil.rmtree(hloc_out)  # ⚠️ borra la corrida anterior de hloc

!cd "{ROOT}" && python run_hloc.py

In [ ]:
import pycolmap, shutil

data = ROOT / "data"
# ⚠️ limpia el layout anterior (data/images, data/sparse) antes de regenerarlo
for d in (data / "images", data / "sparse", data / "stereo", data / "distorted"):
    if d.exists():
        shutil.rmtree(d)

pycolmap.undistort_images(
    output_path=str(data),
    input_path=str(ROOT / "hloc_out" / "sfm"),
    image_path=str(data / "input"),
    output_type="COLMAP",
)

sparse0 = data / "sparse" / "0"
sparse0.mkdir(parents=True, exist_ok=True)
for f in (data / "sparse").iterdir():
    if f.is_file():
        shutil.move(str(f), sparse0 / f.name)
print(sorted(p.name for p in sparse0.iterdir()))

### Opción B: COLMAP (CPU) — horas en esta máquina

SIFT + matching secuencial con loop detection (`run_colmap.sh`). Produce directamente el layout 3DGS en `data/`. Requiere `vocab_tree.bin` en la raíz. El log queda en pantalla; si lo corres fuera del notebook, mejor `./run_colmap.sh > colmap.log 2>&1 &`.

⚠️ Las etapas de extracción y matching son lentas en CPU — no interrumpas aunque parezca colgado.

In [ ]:
# import shutil
# for d in (ROOT/'data'/'images', ROOT/'data'/'sparse', ROOT/'data'/'stereo', ROOT/'data'/'distorted'):
#     if d.exists():
#         shutil.rmtree(d)  # ⚠️ borra la reconstrucción anterior
#
# !cd "{ROOT}" && bash run_colmap.sh

## 4. Entrenar 3DGS

⚠️ Sobrescribe `output/dko3d/`. Cambia `-m` si quieres conservar el modelo anterior (p. ej. `output/dko3d_v2`).

In [ ]:
MODEL_DIR = ROOT / "output" / "dko3d"

!cd "{ROOT}/gaussian-splatting" && python train.py \
    -s "{ROOT}/data" -m "{MODEL_DIR}" \
    --iterations {ITERATIONS} --save_iterations {ITERATIONS} --test_iterations -1

## 5. Resultado

El modelo queda en `output/dko3d/point_cloud/iteration_*/point_cloud.ply`. Ábrelo en [SuperSplat](https://playcanvas.com/supersplat/editor) (arrastra el `.ply` al navegador) o con el visor SIBR del repo de Inria.

In [ ]:
for ply in sorted((ROOT / "output").rglob("point_cloud.ply")):
    print(ply, f"({ply.stat().st_size / 1e6:.1f} MB)")